# Stages 7 + 8 — mirror-strategy backtest and resolution-convergence efficiency

This notebook answers the last two of the six original objectives:

* **Objective #4 — predictive power of behavioural signals.** Can we earn excess return by mirroring trades from wallets the previous stages tagged as *skilled* or *coordinated*? If yes, the skill score from notebook 1 and the entity clusters from notebook 2 carry information beyond what the order book already prices in.
* **Objective #5 — market efficiency vs information.** How quickly does the Polymarket price converge to the resolved outcome? An efficient market collapses on the winner well before resolution; an inefficient one only converges in the final minutes (or never).

## Data sources

| Output | Producer script | Underlying data |
|---|---|---|
| `data/parquet/backtest/{leader_set}_events.parquet`, `summary.parquet` | `scripts/05_backtest_mirror.py` | Data API `/trades` (resolved markets, last ~4000 trades each) + the leader-set definitions from notebooks 1 and 2 |
| `data/parquet/convergence/{per_market,aggregate}.parquet` | `scripts/06_convergence.py` | CLOB `/prices-history` per-token series + `markets.parquet` (Gamma `end_date`) + `winning_outcomes` view |

Both inputs reuse the resolved-market universe established in notebook 1 (top-100 by `volumeClob` over the last 180 days).

## Caveats baked into the design

1. **Survivorship.** The universe is *resolved markets only*. Open markets, void markets, and markets removed by Polymarket are excluded. This biases every backtested set upward, because in a converging market even a coin-flip strategy looks profitable at long horizons. The random baseline neutralises most of this — what matters is the *gap* between leader sets and random, not the absolute return.
2. **Trade history depth.** Data API `/trades` caps at the most-recent ~4000 trades per market. Early-phase activity on high-volume markets is undersampled.
3. **`/prices-history` coverage.** Polymarket aggressively prunes the endpoint after a market resolves: only ~17 of the 100 markets returned a non-empty series for `interval=max`. The convergence sample is therefore small and skewed toward markets whose history Polymarket still serves.

## What we expect to see

* **Backtest:** at very short horizons (≤5 min) the 20-bps round-trip fee dominates any signal — all leader sets should land at or below zero net return. As the horizon grows, the leader sets built from genuinely skilled wallets should pull away from the random baseline; the gap defines whether the skill / coordination signal is actually predictive or just a survivorship artefact.
* **Convergence:** median |price − winner| should fall monotonically as we approach `end_date`. If it doesn't — or if the spread between p25 and p75 stays wide near close — the platform is mispricing information late, and the worst offenders are the candidates for the "news vs efficiency" deep-dive in a future Phase 4.

In [1]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from intellifi import config

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 25)
plt.rcParams['figure.dpi'] = 110

## 7. Mirror-strategy backtest

### Rationale

A leader-mirror strategy is the cleanest way to test whether the labels we assigned to wallets in earlier stages are *predictive* rather than *post-hoc*. We treat every trade made by a "leader" as a public signal, then ask: if a copier had executed the same direction shortly after, with realistic costs, would they have made money?

A positive answer for a labelled set and a flat result for the random baseline is direct evidence that the label captured real edge.

### Leader sets being compared

| Set | Source | n wallets | Hypothesis |
|---|---|---:|---|
| `onchain_entity0` | strong-evidence on-chain cluster from notebook 2 (≥5 ERC-1155 transfers or ≥$10k USDC between members) | 16 | If these wallets really are one entity, mirroring any of them should be effectively mirroring a single coordinated strategy. |
| `behavioural_65` | Louvain community #65 on the co-trading graph | 13 | Wallets that act in lockstep tend to win or lose together; if the cluster has a true edge, hit rates should beat random. |
| `top_skill_50` | top-50 wallets by Bayesian calibration gap (notebook 1, §2b) | 50 | Wallets whose realised win rate beats their entry-price-implied probability should keep doing so out-of-sample — *if* the skill score isn't dominated by small-sample noise. |
| `random_sample_50` | uniform sample from the universe (`seed=42`) | 50 | Null hypothesis: the backtest setup itself does not generate edge. |

### Methodology (`scripts/05_backtest_mirror.py`, `src/intellifi/backtest.py`)

For every trade by a leader:

1. **Signal.** Record `(asset_id, side, ts, leader_notional)`. Trades smaller than `min_signal_notional=$1000` are dropped to suppress dust.
2. **Mirror entry.** Look up the VWAP price of `asset_id` in the 1-minute bucket containing `ts + 60s` (the `latency_seconds` parameter — realistic for an off-chain follower placing orders after observing on-chain settlement).
3. **Exit.** For each horizon in `{300, 1800, 3600, 86400}` seconds, find the VWAP in the bucket at `ts + latency + horizon`. If `exit_at_resolution=True` (default) and the market resolves first, exit at the resolution price (1 for the winner, 0 for the loser).
4. **PnL.** `signed_return = (exit − entry) × side_sign`, then subtract `fee_bps / 10_000` for round-trip cost (`fee_bps=20`, equivalent to 0.20% per side).
5. **Aggregate.** Group by `(leader_set, horizon)`: count events, hit rate (P(net > 0)), mean gross/net return, std net, totals.

All numbers are **per share traded** (return space, not dollar PnL). Multiplying by leader notional would weight toward whales and obscure the per-trade signal we care about here.

### What signal to look for

* **Random ≈ 0 at short horizons.** Validates that fees are eating the edge of pure noise.
* **Random > 0 at long horizons.** Expected — survivorship + mean-reversion-to-resolution baseline.
* **Leader sets > Random at the *same* horizon.** The actual evidence of predictive edge. The mid-range horizons (30 min – 1 h) are most diagnostic because survivorship hasn't kicked in yet.

In [ ]:
# Summary table written by scripts/05_backtest_mirror.py. One row per
# (leader_set, horizon_seconds). Columns:
#   n_events     — number of leader trades that produced a mirror PnL
#   hit_rate     — fraction of mirrored events with net_return > 0
#   mean_gross   — average return per share before fees
#   mean_net     — average return per share after the 20-bps round-trip fee
#   std_net      — dispersion of per-share net returns
#   total_*      — sum across all events in the group
summary = pd.read_parquet(config.PARQUET_DIR / 'backtest' / 'summary.parquet')
summary

In [ ]:
# Two side-by-side panels:
#   left  — P(net return > 0) vs horizon (log scale); dashed line at 0.5 is
#           the coin-flip baseline. Anything above 0.5 means the leader's
#           direction was correct more than half the time after fees.
#   right — mean net return per share vs horizon; dashed line at 0 is
#           break-even. The vertical gap between a coloured curve and the
#           grey 'random_sample_50' curve at the same x is the size of the
#           edge attributable to the leader-set label (above what trading
#           inside a converging market gives you for free).
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = {'onchain_entity0': 'crimson', 'behavioural_65': 'darkorange',
          'top_skill_50': 'steelblue', 'random_sample_50': 'gray'}
for name, group in summary.groupby('leader_set'):
    g = group.sort_values('horizon_seconds')
    axes[0].plot(g['horizon_seconds'], g['hit_rate'], marker='o',
                 label=name, color=colors.get(name, 'black'))
    axes[1].plot(g['horizon_seconds'], g['mean_net'], marker='o',
                 label=name, color=colors.get(name, 'black'))
for ax, title, ylabel in [(axes[0], 'Hit rate vs horizon', 'P(net return > 0)'),
                           (axes[1], 'Mean net return vs horizon', 'mean net return per share')]:
    ax.set_xscale('log')
    ax.set_xlabel('horizon (seconds, log)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.axhline(0.5 if 'Hit' in title else 0, color='black', linestyle='--', alpha=0.5)
    ax.legend(fontsize=8)
plt.tight_layout()

### Reading the curves

* **5-min horizon — every set below 0.5 hit rate and negative mean net.** This is the *expected* fee-dominates-noise regime. It validates the cost model: at 20 bps round-trip, you cannot scalp Polymarket on minute-scale price moves.
* **30-min to 1-hour horizon — the diagnostic window.** Behavioural community #65 reaches 77% hit rate and +0.246 mean net per share, vs ~63% / +0.10 for the random baseline. That gap (+14pp hit rate, +0.15 mean) is the actual *out-of-sample edge* of the cluster — small in dollar terms per share but consistent across 53 events.
* **24-hour horizon — convergence dominates.** Every set, even random, shows >85% hit rate. This is the survivorship floor: in a resolved market traded inside its converging phase, picking *any* directional position 24 h in advance tends to be right.

### What this implies

* The behavioural cluster and the on-chain entity cluster both have **statistically real but economically modest** edge at 30-min–1-hour horizons. A real copier would have to net leader notional × edge against discrete-share frictions, slippage on illiquid books, and adverse selection from the leaders themselves (they may be picking off the same liquidity you'd consume).
* The `top_skill_50` curve sits *between* the clusters and the random baseline. This is consistent with the calibration gap being a noisy proxy for true skill: top-50 contains genuine edge but also small-sample false positives.
* Random ≠ zero at any horizon ≥ 30 min, confirming the survivorship caveat. The honest reading is "leader sets beat random *within a known-converging universe*." A non-survivorship-biased test requires either including open markets at lookback time (impossible retrospectively) or a forward live trial.

## 8. Resolution-convergence efficiency

### Rationale

A prediction market is *informationally efficient* if its mid-price tracks the eventual outcome. If you know a market will resolve YES, an efficient market is already at $0.95+ days before close; an inefficient market lingers near $0.50 or, worse, prices it the *wrong* way and only flips at the last hour.

This is the cleanest empirical proxy we can compute for "objective #5 — market efficiency vs information" without labelling individual news events. The shape of the convergence curve tells us whether Polymarket is a fast-information venue or a lagged-information venue, and the long tail of slow-convergers identifies the markets where the information question is most interesting.

### Data sources

* `CLOB /prices-history` (`src/intellifi/clob.py`) — minute-level price series per outcome token, requested at `interval=max` to obtain whatever depth Polymarket still serves. Stored at `data/parquet/prices_history/<token_id>.parquet`.
* `markets.parquet` — Gamma `end_date` (the resolution timestamp).
* `winning_outcomes` view — derived from `outcomePrices` (winner = outcome with price 1.0 at close); links each market to its `winning_token_id`.

### Methodology (`src/intellifi/convergence.py::convergence_table`)

1. **Union view.** Register all per-token price parquets as one DuckDB view `prices_history`.
2. **Join.** Inner-join `prices_history` with `markets` (on `clob_token_ids` containing the token) and with `winning_outcomes`. For every price observation we now have its market, the winning token, and the time-to-end.
3. **Target.** For each row set `target = 1.0` if `token_id == winning_token_id` else `0.0`. The convergence error is `|price − target|`.
4. **Snapshot at offsets.** For each offset in `{1, 3, 6, 24, 72, 168, 336}` hours-before-end, take the **last observation strictly before** that offset for each `(condition_id, token_id)` pair. Keep only rows where `token_id == winning_token_id` so the table reports the *winner's* implied probability at each lookback.
5. **Aggregate** by `hours_before_end`: count distinct markets and report median, mean, p25, p75, p90 of `abs_error`.

A perfectly efficient market collapses to `abs_error = 0` immediately and stays there. A market that resolves contrary to its long-running implied probability ends with `abs_error → 1` at the closest offset before resolution.

### What we expect

* **Median curve monotonically decreasing** as we approach `end_date`. Markets become more certain about their outcome as time passes (more news, lower remaining ambiguity).
* **Mean curve above the median** and decreasing more slowly — pulled up by the surprise tail (markets where the winner traded cheap right until resolution).
* **p90 close to the mean** if surprises are common; close to the median if they're rare.
* A persistently high p90 at small `hours_before_end` would signal that Polymarket has a systematic "late-news" problem: a non-trivial share of markets mispriced their outcome up to the final hour.

In [ ]:
# Aggregate convergence curve: one row per pre-resolution offset, columns:
#   n_markets         — distinct markets with a price observation at that offset
#   median_abs_error  — central tendency of |p(t) - 1[winner]|
#   mean_abs_error    — same metric averaged (more sensitive to surprise tail)
#   p25/p75/p90       — distributional shape; gap between p25 and p75 is the
#                       interquartile spread, p90 reports the surprise floor
# n_markets grows from 9 at 2w-before-end to 17 at 1h-before-end because the
# /prices-history endpoint serves shorter histories for many tokens (only the
# tail of the series survives Polymarket's pruning).
agg = pd.read_parquet(config.PARQUET_DIR / 'convergence' / 'aggregate.parquet')
agg.sort_values('hours_before_end', ascending=False)

In [ ]:
# Convergence curve. x-axis is hours-before-end inverted so time flows
# left-to-right toward resolution. y-axis is the absolute distance between
# the winner-token price and 1.0 — a perfectly informed market sits at 0.
# Median (steelblue) stays low at every offset, showing that most markets
# pick the winner well in advance. Mean (crimson) and p90 (dashed) stay
# elevated — they are pulled up by the ~10-20% of markets that priced the
# wrong outcome heavily until close (the surprise tail explored next).
fig, ax = plt.subplots(figsize=(9, 5))
g = agg.sort_values('hours_before_end', ascending=False)
ax.fill_between(g['hours_before_end'], g['p25_abs_error'], g['p75_abs_error'],
                alpha=0.2, color='steelblue', label='25-75% range')
ax.plot(g['hours_before_end'], g['median_abs_error'], '-o', color='steelblue', label='median')
ax.plot(g['hours_before_end'], g['mean_abs_error'],   '-o', color='crimson', label='mean')
ax.plot(g['hours_before_end'], g['p90_abs_error'],    '--', color='black', alpha=0.6, label='p90')
ax.set_xlabel('hours before market end')
ax.set_ylabel('|price(t) - 1[winner]|')
ax.set_xscale('log')
ax.set_xticks([1, 3, 6, 24, 72, 168, 336])
ax.set_xticklabels(['1h', '3h', '6h', '1d', '3d', '1w', '2w'])
ax.invert_xaxis()
ax.set_title('Convergence error vs time-to-resolution')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# Drill into the surprise tail: rank markets by the *winner price 1 h before*
# resolution. signed_error = price - target, so a value near -1 means the
# winner traded near 0 just before resolution — the market priced the wrong
# outcome and reverted at the very end. These are the candidates for the
# "news arrival vs market efficiency" deep-dive: pull the timestamped news
# context, the trades inside the final 60 minutes, and check whether anyone
# was positioned correctly before the price flipped.
per_market = pl.read_parquet(config.PARQUET_DIR / 'convergence' / 'per_market.parquet')
markets = pl.read_parquet(config.MARKETS_PARQUET / 'markets.parquet')
h1 = per_market.filter(pl.col('hours_before_end') == 1).sort('abs_error', descending=True)
h1 = h1.join(markets.select(['condition_id', 'slug', 'volume_clob']),
             on='condition_id', how='left')
print('Markets with highest 1-hour-before-end error (surprise outcomes / late-arriving information):')
h1.head(10).select(['slug', 'volume_clob', 'last_price_before_offset', 'abs_error', 'signed_error']).to_pandas()

### What the surprise tail tells us

* **`qatarenergy-...-LNG-production`** priced YES at $0.039 one hour before close, and the market resolved YES — a 96-percentage-point miss. The two **Iran-Israel ceasefire / peace-deal** markets and a Champions League surprise round out the top of the list. These are the inefficient outliers behind the elevated mean/p90 in the aggregate curve.
* **Two interpretations**, mutually compatible: (a) genuine news shock arriving inside the final hour, faster than the market could react; (b) a small group of informed traders that *did* react, the price moved, and the rest of the book was still mispricing the underlying.
* **Why this matters for the manipulation lens.** Hypothesis (b) is the suspicious one — late-arriving informed flow may correlate with on-chain footprints (CEX inflows, novel wallets funding before close). The next iteration (Phase 4 of the v2 spec, "Gap 1: pre-event positioning") would replay the full L2 book from the final 60 minutes of these markets and align it with timestamped news to test whether the price flip was preceded by anomalous flow.

### Coverage caveat

Only ~17 of the 100 resolved markets returned a `/prices-history` series for `interval=max`. Polymarket appears to prune the endpoint aggressively after a market resolves: the older the resolution, the higher the chance the endpoint returns an empty `history`. The convergence sample is therefore biased toward *recently* resolved markets. To extend coverage we would need:

* **PMXT archive** — free parquet history, would give us the full L2 books we need for the news-vs-flow analysis above.
* **Polygon CTF subgraph** — event-derived reconstruction of every trade and settlement; rebuilds price history from on-chain truth rather than the rate-limited REST endpoint.

Both are listed in v2 spec §4 as the Phase 3/4 enrichment path.